# STT Evaluation (WER) – Jupyter Notebook

This notebook reproduces the behavior of:

```bash
python wer.py gt.txt predict.txt allowed_replacements.txt allowed_insertions.txt allowed_deletions.txt out_stat.txt
```

It embeds the full Python implementation and provides a `run_wer(...)` function to run evaluation inside Jupyter.


## 1. Configure input/output paths

Update the file names below if needed.

- `gt_file`: ground truth text (normalized)
- `predict_file`: STT output text (normalized)
- `allowed_replacements_file`: CSV-like pairs `ref,pred` per line
- `allowed_insertions_file`: one allowed insertion phrase per line
- `allowed_deletions_file`: one allowed deletion phrase per line
- `out_stat_file`: where the report will be written


In [1]:
gt_file = "gt.txt"
predict_file = "predict.txt"
allowed_replacements_file = "allowed_replacements.txt"
allowed_insertions_file = "allowed_insertions.txt"
allowed_deletions_file = "allowed_deletions.txt"
out_stat_file = "out_stat.txt"


## 2. WER implementation (embedded from `wer.py`)
The cell below contains the full code: file utilities, the alignment-based `AccuracyStatistics` class,
and a `run_wer(...)` function that behaves like your `__main__`.


In [2]:
from typing import List, Tuple, Optional
import codecs

def my_reader(in_path):
    fid = codecs.open(in_path, 'r', 'utf-8-sig')
    return fid

def my_appender(out_path):
    fid = codecs.open(out_path, 'a', 'utf-8')
    return fid

def my_writer(out_path):
    fid = codecs.open(out_path, 'w', 'utf-8')
    return fid

def write_line(fid, s):
    fid.write(s + "\n")

def continuous_replace(src, dest, inString):
    prevString = inString
    while True:
        inString = inString.replace(src, dest)
        if inString == prevString:
            break
        prevString = inString
    return inString


class AccuracyStatistics:
    def __init__(self,
                 allowed_replacements: List[Tuple[str, str]],
                 allowed_insertions: List[str],
                 allowed_deletions: List[str]):
        # Copy allowed substitutions both directions
        self._allowed_replacements = []
        for first, second in allowed_replacements:
            self._allowed_replacements.append((first, second))
            self._allowed_replacements.append((second, first))

        self._allowed_insertions = list(allowed_insertions)
        self._allowed_deletions = list(allowed_deletions)

        # Error counters
        self._num_matches = 0
        self._num_substitutions = 0
        self._num_insertions = 0
        self._num_deletions = 0

        self._substitution_counter = {}
        self._insertion_counter = {}
        self._deletion_counter = {}

        # Work matrices for alignment
        self._mat_dim = 0
        self._scores_mat = None
        self._ops_mat = None

    def matches(self) -> int:
        return self._num_matches

    def substitutions(self) -> int:
        return self._num_substitutions

    def insertions(self) -> int:
        return self._num_insertions

    def deletions(self) -> int:
        return self._num_deletions

    def word_error_rate(self) -> float:
        denominator = self._num_matches + self._num_substitutions + self._num_deletions
        if denominator == 0:
            return 0
        numerator = self._num_substitutions + self._num_insertions + self._num_deletions
        return numerator / denominator

    def recall(self) -> float:
        if self._num_matches == 0:
            return 0
        denominator = self._num_matches + self._num_substitutions + self._num_deletions
        return self._num_matches / denominator

    def precision(self) -> float:
        if self._num_matches == 0:
            return 0
        denominator = self._num_matches + self._num_substitutions + self._num_insertions
        return self._num_matches / denominator

    def f1_score(self) -> float:
        if self._num_matches == 0:
            return 0
        rec = self.recall()
        prec = self.precision()
        return (2 * rec * prec) / (rec + prec)

    def substitution_counters(self) -> List[Tuple[str, str, int]]:
        return [(texts[0], texts[1], count) for texts, count in sorted(
            self._substitution_counter.items(), key=lambda pair: (-pair[1], pair[0])
        )]

    def insertion_counters(self) -> List[Tuple[str, int]]:
        return [(text, count) for text, count in sorted(
            self._insertion_counter.items(), key=lambda pair: (-pair[1], pair[0])
        )]

    def deletion_counters(self) -> List[Tuple[str, int]]:
        return [(text, count) for text, count in sorted(
            self._deletion_counter.items(), key=lambda pair: (-pair[1], pair[0])
        )]

    def accumulate(self, reference_words: List[str], transcription_words: List[str]):
        aligned_pairs = self.__global_alignment(reference_words, transcription_words)

        aligned_len = len(aligned_pairs)
        start_ind = 0

        while start_ind < aligned_len:
            next_match_ind = start_ind
            while next_match_ind < aligned_len:
                ref_word, trans_word = aligned_pairs[next_match_ind]
                if (ref_word is not None) and (trans_word is not None) and ref_word == trans_word:
                    break
                next_match_ind += 1

            if next_match_ind == start_ind:
                self._num_matches += 1
                start_ind += 1
                continue

            mismatched_ref_words = []
            mismatched_trans_words = []
            n_subs = 0
            n_inss = 0
            n_dels = 0

            for ref_word, trans_word in aligned_pairs[start_ind: next_match_ind]:
                if ref_word is None:
                    mismatched_trans_words.append(trans_word)
                    n_inss += 1
                elif trans_word is None:
                    mismatched_ref_words.append(ref_word)
                    n_dels += 1
                else:
                    mismatched_ref_words.append(ref_word)
                    mismatched_trans_words.append(trans_word)
                    n_subs += 1

            if len(mismatched_ref_words) == 0:
                inserted_text = ' '.join(mismatched_trans_words)
                if inserted_text not in self._allowed_insertions:
                    self._insertion_counter[inserted_text] = self._insertion_counter.get(inserted_text, 0) + 1
                    self._num_insertions += n_inss

            elif len(mismatched_trans_words) == 0:
                deleted_text = ' '.join(mismatched_ref_words)
                if deleted_text not in self._allowed_deletions:
                    self._deletion_counter[deleted_text] = self._deletion_counter.get(deleted_text, 0) + 1
                    self._num_deletions += n_dels

            else:
                ref_text = ' '.join(mismatched_ref_words)
                trans_text = ' '.join(mismatched_trans_words)
                key_pair = (ref_text, trans_text)

                if key_pair not in self._allowed_replacements:
                    self._substitution_counter[key_pair] = self._substitution_counter.get(key_pair, 0) + 1
                    self._num_substitutions += n_subs
                    self._num_insertions += n_inss
                    self._num_deletions += n_dels
                else:
                    self._num_matches += 1

            start_ind = next_match_ind

    def __global_alignment(self, reference_words: List[str], transcription_words: List[str]) -> List[Tuple[Optional[str], Optional[str]]]:
        op_null = 0
        op_pair = 1
        op_insertion = 2
        op_deletion = 3

        match_weight = 2.0
        substitution_weight = -1.0
        insertion_weight = -0.75
        deletion_weight = -0.75

        num_ref_words = len(reference_words)
        num_trans_words = len(transcription_words)

        if max(num_ref_words, num_trans_words) >= self._mat_dim:
            self._mat_dim = 1 + max(num_ref_words, num_trans_words)
            self._scores_mat = [[0.0] * self._mat_dim for _ in range(self._mat_dim)]
            self._ops_mat = [[0] * self._mat_dim for _ in range(self._mat_dim)]

        self._scores_mat[0][0] = 0.0
        self._ops_mat[0][0] = op_null

        for j in range(num_trans_words):
            self._scores_mat[0][j + 1] = self._scores_mat[0][j] + insertion_weight
            self._ops_mat[0][j + 1] = op_insertion

        for i in range(num_ref_words):
            self._scores_mat[i + 1][0] = self._scores_mat[i][0] + deletion_weight
            for j in range(num_trans_words):
                if reference_words[i] == transcription_words[j]:
                    max_score = self._scores_mat[i][j] + match_weight
                else:
                    max_score = self._scores_mat[i][j] + substitution_weight
                best_op = op_pair

                deletion_score = self._scores_mat[i][j + 1] + deletion_weight
                if deletion_score > max_score:
                    max_score = deletion_score
                    best_op = op_deletion

                insertion_score = self._scores_mat[i + 1][j] + insertion_weight
                if insertion_score > max_score:
                    max_score = insertion_score
                    best_op = op_insertion

                self._scores_mat[i + 1][j + 1] = max_score
                self._ops_mat[i + 1][j + 1] = best_op

        aligned_pairs = []
        curr_i = num_ref_words
        curr_j = num_trans_words

        while curr_i > 0 or curr_j > 0:
            op_code = self._ops_mat[curr_i][curr_j]
            if op_code == op_pair:
                aligned_pairs.append((reference_words[curr_i - 1], transcription_words[curr_j - 1]))
                curr_i -= 1
                curr_j -= 1
            elif op_code == op_insertion:
                aligned_pairs.append((None, transcription_words[curr_j - 1]))
                curr_j -= 1
            elif op_code == op_deletion:
                aligned_pairs.append((reference_words[curr_i - 1], None))
                curr_i -= 1
            else:
                break

        aligned_pairs.reverse()
        return aligned_pairs


def _read_words_list(path: str) -> List[str]:
    words = []
    r = my_reader(path)
    for line in r:
        line = line.strip()
        if not line:
            continue
        line = line.replace("<unk>", "")
        line = continuous_replace("  ", " ", line)
        words.extend(line.split())
    r.close()
    return words


def _read_allowed_replacements(path: str) -> List[Tuple[str, str]]:
    allowed_r = []
    r = my_reader(path)
    for line in r:
        line = line.strip()
        if not line:
            continue
        a = line.split(",", 1)
        if len(a) != 2:
            raise ValueError(f"Bad replacements line (expected 'ref,pred'): {line}")
        allowed_r.append((a[0].strip(), a[1].strip()))
    r.close()
    return allowed_r


def _read_allowed_list(path: str) -> List[str]:
    allowed = []
    r = my_reader(path)
    for line in r:
        line = line.strip()
        if not line:
            continue
        allowed.append(line)
    r.close()
    return allowed


def run_wer(gt_file: str,
            predict_file: str,
            allowed_replacements_file: str,
            allowed_insertions_file: str,
            allowed_deletions_file: str,
            out_stat_file: str) -> AccuracyStatistics:
    ref_list = _read_words_list(gt_file)
    obs_list = _read_words_list(predict_file)

    allowed_r = _read_allowed_replacements(allowed_replacements_file)
    allowed_i = _read_allowed_list(allowed_insertions_file)
    allowed_d = _read_allowed_list(allowed_deletions_file)

    stats = AccuracyStatistics(
        allowed_replacements=allowed_r,
        allowed_insertions=allowed_i,
        allowed_deletions=allowed_d
    )

    stats.accumulate(ref_list, obs_list)

    w = my_writer(out_stat_file)

    write_line(w, 'M = %d, S = %d, I = %d, D = %d' % (
        stats._num_matches, stats._num_substitutions, stats._num_insertions, stats._num_deletions
    ))
    write_line(w, 'WER = %.1f%%, recall = %.1f%%, precision = %.1f%%, F1-score = %.1f%%' % (
        100 * stats.word_error_rate(),
        100 * stats.recall(),
        100 * stats.precision(),
        100 * stats.f1_score()
    ))

    write_line(w, 'Substitutions:')
    for ref, trans, count in stats.substitution_counters():
        write_line(w, '    %s  <->  %s  : %d occurrence(s)' % (ref, trans, count))

    write_line(w, 'Insertions:')
    for trans, count in stats.insertion_counters():
        write_line(w, '    %s  : %d occurrence(s)' % (trans, count))

    write_line(w, 'Deletions:')
    for ref, count in stats.deletion_counters():
        write_line(w, '    %s  : %d occurrence(s)' % (ref, count))

    w.close()
    return stats


## 3. Run evaluation

This cell runs evaluation and prints a short summary. The detailed report is written to `out_stat_file`.

In [3]:
stats = run_wer(
    gt_file=gt_file,
    predict_file=predict_file,
    allowed_replacements_file=allowed_replacements_file,
    allowed_insertions_file=allowed_insertions_file,
    allowed_deletions_file=allowed_deletions_file,
    out_stat_file=out_stat_file,
)

print(f"M = {stats.matches()}, S = {stats.substitutions()}, I = {stats.insertions()}, D = {stats.deletions()}")
print(f"WER = {100*stats.word_error_rate():.1f}%, recall = {100*stats.recall():.1f}%, "
      f"precision = {100*stats.precision():.1f}%, F1-score = {100*stats.f1_score():.1f}%")
print(f"\nReport written to: {out_stat_file}")


M = 2772, S = 82, I = 20, D = 23
WER = 4.3%, recall = 96.4%, precision = 96.5%, F1-score = 96.4%

Report written to: out_stat.txt


## 4. Inspect the report

This cell prints the contents of `out_stat_file`.

In [4]:
with open(out_stat_file, "r", encoding="utf-8") as f:
    print(f.read())


M = 2772, S = 82, I = 20, D = 23
WER = 4.3%, recall = 96.4%, precision = 96.5%, F1-score = 96.4%
Substitutions:
    aimee  <->  amy  : 2 occurrence(s)
    an  <->  and  : 2 occurrence(s)
    doctor kean  <->  dr keene  : 2 occurrence(s)
    in  <->  and  : 2 occurrence(s)
    this  <->  the  : 2 occurrence(s)
    a  <->  one  : 1 occurrence(s)
    a curse  <->  occurse  : 1 occurrence(s)
    a thesaurus  <->  it the soros  : 1 occurrence(s)
    allow  <->  it allows  : 1 occurrence(s)
    and  <->  an  : 1 occurrence(s)
    and  <->  in  : 1 occurrence(s)
    and  <->  now  : 1 occurrence(s)
    antonyms  <->  anthonyms  : 1 occurrence(s)
    at  <->  as  : 1 occurrence(s)
    bushel  <->  bushle  : 1 occurrence(s)
    cast  <->  cash  : 1 occurrence(s)
    circumnavigate the presumed pitfalls  <->  circum navigate to presume pitballs  : 1 occurrence(s)
    d  <->  de  : 1 occurrence(s)
    deficiencies  <->  efficiencies  : 1 occurrence(s)
    disservice  <->  service  : 1 occurrence(